In [1]:
!pip install flask flask-ngrok pyngrok requests

In [2]:
OPENROUTER_API_KEY = "API-KEY"

In [10]:
import requests
import threading
from flask import Flask, request, jsonify

app = Flask(__name__)

MODEL = "openai/gpt-oss-120b:free"

def ask_ai(topic):
    prompt = f"""
You are a helpful study assistant for a computer science student.

The student wants to learn about: {topic}

Give them:
1. A simple, clear explanation (3-4 sentences)
2. One real-world example to understand it better
3. Three quiz questions to test their understanding (with answers)

Format your response clearly with these exact headers:
📖 EXPLANATION:
💡 EXAMPLE:
❓ QUIZ QUESTIONS:
"""
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt}]
        }
    )
    result = response.json()
    return result['choices'][0]['message']['content']


HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>KOGUL Study Assistant</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }

    body {
      font-family: 'Segoe UI', sans-serif;
      background: #0f0f0f;
      color: #f0f0f0;
      min-height: 100vh;
      display: flex;
      align-items: center;
      justify-content: center;
      padding: 20px;
    }

    .container {
      background: #1a1a1a;
      border: 1px solid #2e2e2e;
      border-radius: 16px;
      padding: 40px;
      width: 100%;
      max-width: 680px;
    }

    h1 {
      font-size: 24px;
      font-weight: 600;
      margin-bottom: 6px;
      color: #ffffff;
    }

    p.sub {
      font-size: 14px;
      color: #888;
      margin-bottom: 28px;
    }

    input {
      width: 100%;
      padding: 14px 16px;
      border-radius: 10px;
      border: 1px solid #333;
      background: #111;
      color: #f0f0f0;
      font-size: 15px;
      outline: none;
      margin-bottom: 14px;
      transition: border 0.2s;
    }

    input:focus { border-color: #555; }

    button {
      width: 100%;
      padding: 14px;
      background: #ffffff;
      color: #000000;
      font-size: 15px;
      font-weight: 600;
      border: none;
      border-radius: 10px;
      cursor: pointer;
      transition: background 0.2s;
    }

    button:hover { background: #e0e0e0; }
    button:disabled { background: #444; color: #888; cursor: not-allowed; }

    .result {
      margin-top: 28px;
      background: #111;
      border: 1px solid #2e2e2e;
      border-radius: 12px;
      padding: 24px;
      display: none;
    }

    .result.visible { display: block; }

    .result pre {
      white-space: pre-wrap;
      font-family: 'Segoe UI', sans-serif;
      font-size: 14px;
      line-height: 1.8;
      color: #d0d0d0;
    }

    .loader {
      text-align: center;
      color: #888;
      font-size: 14px;
      margin-top: 20px;
      display: none;
    }

    .loader.visible { display: block; }

    .badge {
      display: inline-block;
      font-size: 11px;
      padding: 3px 10px;
      background: #1e1e1e;
      border: 1px solid #333;
      border-radius: 99px;
      color: #aaa;
      margin-bottom: 20px;
    }
  </style>
</head>
<body>
  <div class="container">
    <span class="badge">Powered by OpenRouter · GPT-OSS 120B</span>
    <h1>AI Study Assistant</h1>
    <p class="sub">Type any topic. Get a simple explanation, a real example, and quiz questions.</p>

    <input type="text" id="topicInput" placeholder="e.g. Recursion, DBMS, Binary Search, OOP..." />
    <button id="askBtn" onclick="askAI()">Ask AI</button>

    <div class="loader" id="loader">⏳ Thinking... this may take a few seconds</div>

    <div class="result" id="result">
      <pre id="resultText"></pre>
    </div>
  </div>

  <script>
    async function askAI() {
      const topic = document.getElementById('topicInput').value.trim();
      if (!topic) return;

      const btn = document.getElementById('askBtn');
      const loader = document.getElementById('loader');
      const result = document.getElementById('result');
      const resultText = document.getElementById('resultText');

      btn.disabled = true;
      btn.textContent = 'Asking AI...';
      loader.classList.add('visible');
      result.classList.remove('visible');

      try {
        const response = await fetch('/ask', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ topic: topic })
        });

        const data = await response.json();

        if (data.result) {
          resultText.textContent = data.result;
          result.classList.add('visible');
        } else {
          resultText.textContent = 'Something went wrong. Please try again.';
          result.classList.add('visible');
        }
      } catch (err) {
        resultText.textContent = 'Error connecting to server. Please try again.';
        result.classList.add('visible');
      }

      btn.disabled = false;
      btn.textContent = 'Ask AI';
      loader.classList.remove('visible');
    }

    document.getElementById('topicInput').addEventListener('keypress', function(e) {
      if (e.key === 'Enter') askAI();
    });
  </script>
</body>
</html>
"""

@app.route('/')
def home():
    return HTML_PAGE

@app.route('/ask', methods=['POST'])
def ask():
    data = request.get_json()
    topic = data.get('topic', '')
    if not topic:
        return jsonify({'error': 'No topic provided'}), 400
    result = ask_ai(topic)
    return jsonify({'result': result})


# Run Flask in a background thread so Colab doesn't freeze
threading.Thread(target=lambda: app.run(port=5000)).start()

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [9]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5000)"))

https://5000-m-s-kkb-usc1c2-1aukq973pxros-c.us-central1-2.prod.colab.dev
